# Orion-0.6B v0.1 — Kaggle T4 LoRA Fine-Tuning & Evaluation

Fine-tunes **`Qwen/Qwen3-0.6B`** with LoRA SFT on reproducible Bangladesh NASA MERRA-2 trend data and Orion's 5 allowlisted backend tools.

### Workflow
1. Verify Kaggle CUDA GPU (`T4`) & Node.js availability.
2. Clone/update `feat/orion-model-training` (or attach dataset/repo).
3. Install `training/requirements-kaggle.txt`.
4. Build & validate leak-free `train` / `val` / `test` splits.
5. Run LoRA SFT (`training/train_lora.py`) and merge optional 16-bit artifact.
6. Evaluate Base `Qwen/Qwen3-0.6B` vs. Fine-tuned `Orion-0.6B v0.1` (`training/evaluate.py`).
7. Upload evaluated adapter/merged weights to a **private** Hugging Face repository using Kaggle Secrets (`HF_TOKEN`).

## 1. Environment & GPU Verification
Confirm CUDA GPU memory (Kaggle T4 = 16 GB) and verify repository directory.

In [ ]:
!nvidia-smi
import os, subprocess, sys
print('Python:', sys.version)
print('Node:', subprocess.getoutput('node -v || echo missing'))

## 2. Repository Checkout & Dependencies
If running in a fresh Kaggle notebook, clone the repository on `feat/orion-model-training` and install pinned training libraries.

In [ ]:
from pathlib import Path

REPO_DIR = Path('/kaggle/working/orion-space')
if not REPO_DIR.exists():
    !git clone -b feat/orion-model-training https://github.com/MD-Mushfiqur123/orion-space.git /kaggle/working/orion-space
%cd /kaggle/working/orion-space
!pip install -q -r training/requirements-kaggle.txt

## 3. Build & Validate Leak-Free SFT Splits
Generates `train.jsonl`, `val.jsonl`, and `test.jsonl` directly from `analysis/results/t2m_trends_by_cell_month.csv` and `src/agent/bangladeshDistricts.js`, then validates schemas, held-out districts, and scientific traps.

In [ ]:
!python3 training/build_dataset.py
!python3 training/validate_dataset.py
!python3 -m unittest training/test_pipeline.py

## 4. Inspect Split Distributions
Verify language balance (`en`, `bn`, `banglish`) and category coverage before launching GPU training.

In [ ]:
import json
from collections import Counter
from pathlib import Path

for split in ['train', 'val', 'test']:
    rows = [json.loads(line) for line in Path(f'training/artifacts/{split}.jsonl').read_text().splitlines() if line.strip()]
    langs = Counter(r['language'] for r in rows)
    cats = Counter(r['category'] for r in rows)
    print(f'=== {split.upper()} ({len(rows)} rows) ===')
    print('Languages:', dict(langs))
    print('Categories:', dict(cats))

## 5. Run LoRA Fine-Tuning on `Qwen/Qwen3-0.6B`
Uses `training/orion_qwen3_config.json` (`r=16`, `lora_alpha=32`, `enable_thinking=false`) and exports both the LoRA adapter and a merged 16-bit checkpoint for reliable vLLM deployment.

In [ ]:
!python3 training/train_lora.py \
  --config training/orion_qwen3_config.json \
  --train-file training/artifacts/train.jsonl \
  --val-file training/artifacts/val.jsonl \
  --output-dir training/outputs/orion-qwen3-0.6b-lora \
  --merge-output training/outputs/orion-qwen3-0.6b-merged

## 6. Evaluate Base vs. Fine-Tuned `Orion-0.6B v0.1`
Compares raw `Qwen/Qwen3-0.6B` against the trained LoRA adapter on held-out districts (`pirojpur`, `dinajpur`, `coxsbazar`, `sunamganj`) and scientific trap checks (October FDR = 0/34, Annual = 0/34, `unavailable` feed honesty).

In [ ]:
!python3 training/evaluate.py \
  --compare-base \
  --adapter-dir training/outputs/orion-qwen3-0.6b-lora/adapter \
  --out-report training/outputs/eval_report.json

## 7. Review Evaluation Report & Export to Private Hugging Face Repo
Set `HF_TOKEN` in Kaggle Add-ons → Secrets (never hardcode tokens in notebook cells).

In [ ]:
import json, os
from pathlib import Path

report = json.loads(Path('training/outputs/eval_report.json').read_text())
print('Base Model:', report.get('base_model_eval'))
print('Orion-0.6B v0.1:', report.get('fine_tuned_eval'))

# Uncomment when ready to push evaluated weights to your private HF model repo:
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
# from huggingface_hub import HfApi
# api = HfApi(token=os.environ['HF_TOKEN'])
# repo_id = 'YOUR_HF_USERNAME/orion-qwen3-0.6b'
# api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
# api.upload_folder(folder_path='training/outputs/orion-qwen3-0.6b-merged', repo_id=repo_id, commit_message='Orion-0.6B v0.1 merged')
# api.upload_file(path_or_fileobj='training/outputs/eval_report.json', path_in_repo='eval_report.json', repo_id=repo_id)
# api.upload_file(path_or_fileobj='training/outputs/orion-qwen3-0.6b-lora/training_summary.json', path_in_repo='training_summary.json', repo_id=repo_id)